In [1]:
pip install torch transformers[torch] numpy scikit-learn

Note: you may need to restart the kernel to use updated packages.


In [2]:
# Import relevant modules
from transformers import AutoModelForSequenceClassification, AutoTokenizer

/anaconda/envs/azureml_py38_PT_TF/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
# Load a pretrained model and tokenizer (e.g., BERT for sequence classification)
model_name = "bert-base-uncased"
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)
tokenizer = AutoTokenizer.from_pretrained(model_name)

2025-12-10 21:00:50.834046: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [4]:
from datasets import load_dataset
from sklearn.model_selection import train_test_split
import pandas as pd

In [5]:
# Load the IMDb dataset
dataset = load_dataset('imdb')

In [13]:
# Convert dataset to Pandas DataFrame
df = pd.DataFrame(dataset['train'])

# Perform train-test split
train_data, test_data = train_test_split(df, test_size=0.2, random_state=42)

# Tokenize the dataset
def preprocess_function(examples):
    return tokenizer(examples['text'], padding="max_length", truncation=True)

In [15]:
# Convert back to Hugging Face dataset
from datasets import Dataset

In [16]:
train_data = Dataset.from_pandas(train_data)
test_data = Dataset.from_pandas(test_data)

In [17]:
# Apply preprocessing
train_data = train_data.map(preprocess_function, batched=True)
test_data = test_data.map(preprocess_function, batched=True)

Map: 100%|██████████| 5000/5000 [00:02<00:00, 1704.40 examples/s]


In [18]:
import os
import torch
from transformers import Trainer, TrainingArguments, AutoModelForSequenceClassification

In [19]:
# Disable parallelism warning and MLflow logging
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["MLFLOW_TRACKING_URI"] = "disable"
os.environ["HF_MLFLOW_LOGGING"] = "false"

In [20]:
# Ensure CPU usage if no GPU is available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [21]:
# Load a smaller, faster model like DistilBERT
model = AutoModelForSequenceClassification.from_pretrained('distilbert-base-uncased', num_labels=2)
model.to(device)

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


DistilBertForSequenceClassification(
  (distilbert): DistilBertModel(
    (embeddings): Embeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (transformer): Transformer(
      (layer): ModuleList(
        (0-5): 6 x TransformerBlock(
          (attention): DistilBertSdpaAttention(
            (dropout): Dropout(p=0.1, inplace=False)
            (q_lin): Linear(in_features=768, out_features=768, bias=True)
            (k_lin): Linear(in_features=768, out_features=768, bias=True)
            (v_lin): Linear(in_features=768, out_features=768, bias=True)
            (out_lin): Linear(in_features=768, out_features=768, bias=True)
          )
          (sa_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
          (ffn): FFN(
            (dropout): Dropout(p=0.1, inplace=False)


In [22]:
# Use a subset of the dataset to speed up training
train_data = train_data.select(range(1000))  # Select 1000 samples for training
test_data = test_data.select(range(200))     # Select 200 samples for evaluation

In [23]:
# Set up training arguments for faster training
training_args = TrainingArguments(
    output_dir='./results',
    eval_strategy="steps",
    eval_steps=500,
    learning_rate=2e-5,
    per_device_train_batch_size=8,   
    num_train_epochs=1,              
    weight_decay=0,                  
    logging_steps=500,               
    save_steps=1000,                 
    save_total_limit=1,              
    gradient_accumulation_steps=1,   
    fp16=False,                      
    report_to="none",                
)

/anaconda/envs/azureml_py38_PT_TF/lib/python3.10/site-packages/torch/cuda/__init__.py:789: UserWarning: Can't initialize NVML
  warnings.warn("Can't initialize NVML")


In [24]:
# Define the Trainer for fine-tuning
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_data,
    eval_dataset=test_data,
)

In [25]:
# Fine-tune the model
trainer.train()

/anaconda/envs/azureml_py38_PT_TF/lib/python3.10/site-packages/torch/utils/data/dataloader.py:665: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Step,Training Loss,Validation Loss


TrainOutput(global_step=125, training_loss=0.5466298828125, metrics={'train_runtime': 1059.6959, 'train_samples_per_second': 0.944, 'train_steps_per_second': 0.118, 'total_flos': 132467398656000.0, 'train_loss': 0.5466298828125, 'epoch': 1.0})

In [28]:
# Evaluate the model
results = trainer.evaluate()

In [29]:
print(results)

{'eval_loss': 0.3290671408176422, 'eval_runtime': 41.4856, 'eval_samples_per_second': 4.821, 'eval_steps_per_second': 0.603, 'epoch': 1.0}
